# HQ-CELL-00 — Project map and current status
- **semantic cell ID**: `HQ-CELL-00`
- **stage name**: Project map and current status
- **purpose**: Show θ→geometry→x→y and inverse-design target chain without executing it.
- **input**: Official logs, HQ v0.1, IMSTL-007, XREG-v2.7
- **output**: Human-readable project/state map
- **current status**: **READY**
- **applicable source/family**: all sources/families
- **implementation module**: `notebook markdown + stage registry`
- **validation evidence**: CTRL-20260728-M01; IMSTL-007-20260728-002
- **blocking condition**: none
- **related work ID**: `HQ-BLUEPRINT-001`
- **next evidence required**: Keep roadmap synchronized

**Official relationship**

- HQ v0.1 is the official technical integration state and remains untouched.
- HQ Blueprint v0.2 is a development map, not a submission replacement.

```text
θ → geometry G → descriptor x → performance y
target y* → required x* → generation parameter θ* → geometry
```

Generated STL, imported STL and original STP/STEP remain separate routes. LOCKED/PLANNED stages cannot execute.

In [ ]:
# # HQ-CELL-01 — MASTER CONTROLLER
# - semantic cell ID: `HQ-CELL-01`
# - stage name: MASTER CONTROLLER
# - purpose: Expose the sole public research-setting surface.
# - input: User-edited BlueprintController fields
# - output: Typed public configuration
# - current status: READY
# - applicable source/family: all planned routes
# - implementation module: `urp4.hq.v0_2.models.BlueprintController`
# - validation evidence: HQ v0.1 Controller 40/40; blueprint public-field test
# - blocking condition: execution is status-only
# - related work ID: `HQ-BLUEPRINT-001`
# - next evidence required: Future gates may unlock fields individually
from pathlib import Path
import sys
HERE = Path.cwd().resolve()
DELIVERABLE_ROOT = HERE if (HERE / "URP4_1_HQ.ipynb").is_file() else HERE / "URP4-1_DELIVERABLE"
if not (DELIVERABLE_ROOT / "URP4_1_HQ.ipynb").is_file():
    raise FileNotFoundError("Run from URP4-1 project root or URP4-1_DELIVERABLE")
if str(DELIVERABLE_ROOT) not in sys.path:
    sys.path.insert(0, str(DELIVERABLE_ROOT))
from urp4.hq.v0_2 import BlueprintController

# USER EDIT SURFACE — status-only defaults. Unsupported execution switches fail closed.
CONTROLLER = BlueprintController(
    execution_mode="blueprint_status_only",
    geometry_mode="status_only",
    input_file="", input_directory="",
    source_type="imported_stl", model_family="unspecified", generation_parameters={},
    normalize_all_analysis_geometry=True, normalization_target_mm=40.0,
    normalization_method="centered_uniform_bbox_to_40mm", enable_generated_normalization=False,
    slicing_axis="z", pixel_resolution=1000, slice_count=801, slice_spacing_mm=0.05,
    threshold_rule="binary_nonzero_png_readback", connectivity=8, min_component_pixels=2,
    descriptor_scope=("run139_minimal_9_scalar",), candidate_registry_version="XREG-v2.7-TECHNICAL",
    batch_enabled=False, resume_enabled=True, y_target="",
    feature_selection_method="LOCKED", training_method="LOCKED", ensemble_policy="LOCKED",
    enable_y_intake=False, enable_feature_selection=False, enable_training=False,
    enable_ensemble=False, enable_forward_performance=False, enable_inverse_design=False,
    artifact_policy="STREAMING_TEMP_PNG", retain_selected_review_images=True,
    output_directory="outputs/blueprint_status",
)
CONTROLLER


In [ ]:
# # HQ-CELL-02 — Derived config, validation and freeze
# - **semantic cell ID**: `HQ-CELL-02`
# - **stage name**: Derived config, validation and freeze
# - **purpose**: Validate dependencies, KMK312, 40 mm policy and freeze a config hash.
# - **input**: BlueprintController
# - **output**: Frozen status-only config JSON
# - **current status**: **READY**
# - **applicable source/family**: all planned routes
# - **implementation module**: `urp4.hq.v0_2.runtime.validate_and_freeze_blueprint`
# - **validation evidence**: Blueprint verifier fail-closed matrix
# - **blocking condition**: non-status execution
# - **related work ID**: `HQ-BLUEPRINT-001`
# - **next evidence required**: Per-stage implementation permits
from urp4.hq.v0_2 import validate_and_freeze_blueprint, write_frozen_blueprint
FROZEN = validate_and_freeze_blueprint(CONTROLLER, DELIVERABLE_ROOT)
FROZEN_CONFIG_PATH = write_frozen_blueprint(FROZEN, DELIVERABLE_ROOT)
print(FROZEN['status'], FROZEN['run_id'], FROZEN['config_sha256'])
print(FROZEN_CONFIG_PATH)


In [ ]:
# # HQ-CELL-03 — Pipeline stage registry and execution plan
# - **semantic cell ID**: `HQ-CELL-03`
# - **stage name**: Pipeline stage registry and execution plan
# - **purpose**: Render and validate the 22-stage ordered registry.
# - **input**: HQ_BLUEPRINT_V0_2_STAGE_REGISTRY.json
# - **output**: Validated status table
# - **current status**: **READY**
# - **applicable source/family**: all
# - **implementation module**: `urp4.hq.v0_2.runtime.validate_stage_registry`
# - **validation evidence**: Registry/order/semantic-ID verifier
# - **blocking condition**: none
# - **related work ID**: `HQ-BLUEPRINT-001`
# - **next evidence required**: Update registry only with evidence
import pandas as pd
from urp4.hq.v0_2 import load_stage_registry
REGISTRY_PATH = DELIVERABLE_ROOT / 'config/HQ_BLUEPRINT_V0_2_STAGE_REGISTRY.json'
REGISTRY = load_stage_registry(REGISTRY_PATH)
pd.DataFrame(REGISTRY['stages'])[['stage_id','stage_name','status','related_work_id','blocked_by','next_gate']]


In [ ]:
# # HQ-CELL-04 — Input inventory and geometry source router
# - **semantic cell ID**: `HQ-CELL-04`
# - **stage name**: Input inventory and geometry source router
# - **purpose**: Keep generated STL, imported STL and original STP routes explicit and hashed.
# - **input**: File/directory path plus declared source_type
# - **output**: Input manifest with SHA, bbox and route
# - **current status**: **READY**
# - **applicable source/family**: generated_stl; imported_stl; original_stp
# - **implementation module**: `urp4.geometry_io.v0_3 + v0_4; urp4.hq.v0_1`
# - **validation evidence**: HQ v0.1 route smoke 9/9; IMSTL-001
# - **blocking condition**: silent STL→STEP fallback forbidden
# - **related work ID**: `IMSTL-001`
# - **next evidence required**: Direct-STP descriptor adapter if required
from urp4.hq.v0_2 import stage_view
STAGE_04 = stage_view(REGISTRY, 'HQ-CELL-04')
STAGE_04


In [ ]:
# # HQ-CELL-05 — Geometry generation/import
# - **semantic cell ID**: `HQ-CELL-05`
# - **stage name**: Geometry generation/import
# - **purpose**: Inventory family generators and source import capabilities.
# - **input**: Frozen source/generator contract
# - **output**: Geometry artifact plus provenance
# - **current status**: **EXPERIMENTAL**
# - **applicable source/family**: Lattice A/B; TPMS; Voxel; imported STL; original STP
# - **implementation module**: `urp4.generators.*; urp4.geometry_io.*`
# - **validation evidence**: HQ technical full chains; source identities 14/14
# - **blocking condition**: Type B workbook missing; original STP descriptor unavailable
# - **related work ID**: `HQ-GEN-001`
# - **next evidence required**: Type B identity intake; production generator policy
from urp4.hq.v0_2 import stage_view
STAGE_05 = stage_view(REGISTRY, 'HQ-CELL-05')
STAGE_05


In [ ]:
# # HQ-CELL-06 — 40×40×40 mm normalization
# - **semantic cell ID**: `HQ-CELL-06`
# - **stage name**: 40×40×40 mm normalization
# - **purpose**: Apply the project decision to preserve originals and create centered uniform-scale analysis copies.
# - **input**: Source geometry + source bbox
# - **output**: Normalized copy/transform manifest targeting 40 mm
# - **current status**: **EXPERIMENTAL**
# - **applicable source/family**: all analysis geometry
# - **implementation module**: `existing imported normalization; future urp4.geometry normalization service`
# - **validation evidence**: IMSTL-005 imported normalization; HQ audit bbox 41.4074/39.5/38.0
# - **blocking condition**: generated-family normalization regression missing
# - **related work ID**: `HQ-GEOM-001`
# - **next evidence required**: Generated Lattice/TPMS/Voxel normalization implementation and regression
from urp4.hq.v0_2 import stage_view
STAGE_06 = stage_view(REGISTRY, 'HQ-CELL-06')
STAGE_06


In [ ]:
# # HQ-CELL-07 — Geometry QA
# - **semantic cell ID**: `HQ-CELL-07`
# - **stage name**: Geometry QA
# - **purpose**: Record bbox, finite mesh, duplicate/topology/orientation risk and quarantine decision.
# - **input**: Routed geometry
# - **output**: Geometry QA/preflight record
# - **current status**: **READY**
# - **applicable source/family**: generated STL; imported STL; original STP
# - **implementation module**: `urp4.geometry_io.v0_3.source_preflight`
# - **validation evidence**: IMSTL-001; HQ audit
# - **blocking condition**: QA is diagnostic, not automatic scientific approval
# - **related work ID**: `IMSTL-001`
# - **next evidence required**: Family-specific acceptance policies
from urp4.hq.v0_2 import stage_view
STAGE_07 = stage_view(REGISTRY, 'HQ-CELL-07')
STAGE_07


In [ ]:
# # HQ-CELL-08 — Image slicing and artifact policy
# - **semantic cell ID**: `HQ-CELL-08`
# - **stage name**: Image slicing and artifact policy
# - **purpose**: Use traced P1000/Z801 defaults and explicit temporary/retained-image policy.
# - **input**: Qualified geometry route + slicing config
# - **output**: Slice images/readback trace/retention manifest
# - **current status**: **READY**
# - **applicable source/family**: generated STL; screening-qualified imported STL
# - **implementation module**: `urp4.descriptor_service.v0_1.slicing; geometry_io.v0_4`
# - **validation evidence**: RUN-139; IMSTL-004/006/007; HQ full chain
# - **blocking condition**: original STP direct slicing unavailable; all58 not authorized
# - **related work ID**: `IMSTL-008`
# - **next evidence required**: F1 full P1000/Z801; separate broader generalization
from urp4.hq.v0_2 import stage_view
STAGE_08 = stage_view(REGISTRY, 'HQ-CELL-08')
STAGE_08


In [ ]:
# # HQ-CELL-09 — Pixel and connected-component extraction
# - **semantic cell ID**: `HQ-CELL-09`
# - **stage name**: Pixel and connected-component extraction
# - **purpose**: Produce pixel and component tables under traced threshold/CC rules.
# - **input**: Slice/overlay PNG readback
# - **output**: Four primitive tables + QA
# - **current status**: **READY**
# - **applicable source/family**: qualified slice-image routes
# - **implementation module**: `urp4.descriptor_service.v0_1.pixel + component + pipeline`
# - **validation evidence**: RUN-139 58/58; HQ P1000/Z801; IMSTL full traces
# - **blocking condition**: source-route qualification still applies
# - **related work ID**: `RUN-139`
# - **next evidence required**: Descriptor-scope expansion without changing primitive lineage
from urp4.hq.v0_2 import stage_view
STAGE_09 = stage_view(REGISTRY, 'HQ-CELL-09')
STAGE_09


In [ ]:
# # HQ-CELL-10 — LEGACY-PY direct descriptor extraction
# - **semantic cell ID**: `HQ-CELL-10`
# - **stage name**: LEGACY-PY direct descriptor extraction
# - **purpose**: Separate the frozen nine scalar service from the wider LEGACY-PY descriptor inventory.
# - **input**: Primitive tables, point/surface inputs where applicable
# - **output**: Traceable direct descriptor table
# - **current status**: **EXPERIMENTAL**
# - **applicable source/family**: descriptor-specific
# - **implementation module**: `LEGACY-PY inventory + urp4.descriptor_service.v0_1`
# - **validation evidence**: RUN-139 nine scalars; legacy integration evidence
# - **blocking condition**: full formula/population/source lineage not unified
# - **related work ID**: `HQ-DESC-001`
# - **next evidence required**: Inventory and execute every validated direct descriptor
from urp4.hq.v0_2 import stage_view
STAGE_10 = stage_view(REGISTRY, 'HQ-CELL-10')
STAGE_10


In [ ]:
# # HQ-CELL-11 — Candidate descriptor factory
# - **semantic cell ID**: `HQ-CELL-11`
# - **stage name**: Candidate descriptor factory
# - **purpose**: Inventory and selectively compute traceable direct/derived/sensitivity candidates.
# - **input**: Validated raw tables + candidate registry version
# - **output**: Unselected candidate values with lineage
# - **current status**: **EXPERIMENTAL**
# - **applicable source/family**: 58-model frozen raw-table population where applicable
# - **implementation module**: `PRM Grade B/C factory; future HQ candidate adapter`
# - **validation evidence**: XREG-v2.7: 542 candidates, FAST no-y
# - **blocking condition**: not integrated as an HQ execution service; Grade-A gates separate
# - **related work ID**: `XREG-v2.7`
# - **next evidence required**: HQ candidate-service adapter with immutable versioning
from urp4.hq.v0_2 import stage_view
STAGE_11 = stage_view(REGISTRY, 'HQ-CELL-11')
STAGE_11


In [ ]:
# # HQ-CELL-12 — Descriptor QA and registry
# - **semantic cell ID**: `HQ-CELL-12`
# - **stage name**: Descriptor QA and registry
# - **purpose**: Attach formula, population, unit, coverage, missingness, risk and confidence states.
# - **input**: Direct/derived descriptor records
# - **output**: Descriptor registry and x-only QA
# - **current status**: **EXPERIMENTAL**
# - **applicable source/family**: candidate-specific
# - **implementation module**: `XREG/PRM registries; future HQ registry adapter`
# - **validation evidence**: PRM producer/independent QA; x-only redundancy graph
# - **blocking condition**: canonical/primary promotion forbidden without later evidence
# - **related work ID**: `HQ-DESC-002`
# - **next evidence required**: Unified registry schema and source-family applicability audit
from urp4.hq.v0_2 import stage_view
STAGE_12 = stage_view(REGISTRY, 'HQ-CELL-12')
STAGE_12


In [ ]:
# # HQ-CELL-13 — Full unselected X matrix export
# - **semantic cell ID**: `HQ-CELL-13`
# - **stage name**: Full unselected X matrix export
# - **purpose**: Export all computed traceable descriptors before selection without overwriting source X.
# - **input**: Descriptor registry + values
# - **output**: CSV/XLSX/schema/manifest unselected X
# - **current status**: **PLANNED**
# - **applicable source/family**: qualified model population
# - **implementation module**: `planned urp4 descriptor-matrix export`
# - **validation evidence**: XREG long tables exist outside HQ
# - **blocking condition**: HQ export adapter and population contract absent
# - **related work ID**: `HQ-XMAT-001`
# - **next evidence required**: Immutable full-X export implementation/QA
from urp4.hq.v0_2 import stage_view
STAGE_13 = stage_view(REGISTRY, 'HQ-CELL-13')
STAGE_13


In [ ]:
# # HQ-CELL-14 — Experimental y intake and crosswalk
# - **semantic cell ID**: `HQ-CELL-14`
# - **stage name**: Experimental y intake and crosswalk
# - **purpose**: Validate future experimental performance data and provenance.
# - **input**: model/family/replicate/direction/source/y/unit/provenance
# - **output**: Validated y registry and crosswalk
# - **current status**: **LOCKED**
# - **applicable source/family**: future official experiment population
# - **implementation module**: `future urp4.y_intake`
# - **validation evidence**: Schema requirements documented; official new y absent
# - **blocking condition**: official experimental data and permit absent
# - **related work ID**: `HQ-Y-001`
# - **next evidence required**: Doctor-provided model+test results and crosswalk audit
from urp4.hq.v0_2 import stage_view
STAGE_14 = stage_view(REGISTRY, 'HQ-CELL-14')
STAGE_14


In [ ]:
# # HQ-CELL-15 — Feature Selection
# - **semantic cell ID**: `HQ-CELL-15`
# - **stage name**: Feature Selection
# - **purpose**: Run four supplied methods with grouped/nested leakage-safe evaluation after y approval.
# - **input**: Immutable full X + official y + grouped split
# - **output**: Selected-feature registry separate from X
# - **current status**: **LOCKED**
# - **applicable source/family**: output/family-specific
# - **implementation module**: `urp4.training adapters; future selection orchestrator`
# - **validation evidence**: Professor Training code inventory; policy modules
# - **blocking condition**: official y/modeling permit absent
# - **related work ID**: `HQ-FS-001`
# - **next evidence required**: Method crosswalk, nested grouped evaluation and approval
from urp4.hq.v0_2 import stage_view
STAGE_15 = stage_view(REGISTRY, 'HQ-CELL-15')
STAGE_15


In [ ]:
# # HQ-CELL-16 — Training method 1–5
# - **semantic cell ID**: `HQ-CELL-16`
# - **stage name**: Training method 1–5
# - **purpose**: Compare output-specific methods against null/baseline under grouped evaluation.
# - **input**: Selected X, official y, family groups
# - **output**: OOF metrics/models/ledger
# - **current status**: **LOCKED**
# - **applicable source/family**: output/family-specific
# - **implementation module**: `urp4.training.*`
# - **validation evidence**: Training source inventory and fail-closed skeleton
# - **blocking condition**: official y and modeling permit absent
# - **related work ID**: `HQ-TRAIN-001`
# - **next evidence required**: Authorized grouped modeling experiment
from urp4.hq.v0_2 import stage_view
STAGE_16 = stage_view(REGISTRY, 'HQ-CELL-16')
STAGE_16


In [ ]:
# # HQ-CELL-17 — Best model / ensemble
# - **semantic cell ID**: `HQ-CELL-17`
# - **stage name**: Best model / ensemble
# - **purpose**: Choose output-specific models and partial ensemble with uncertainty/holdout evidence.
# - **input**: Approved method results
# - **output**: Ensemble/model registry
# - **current status**: **LOCKED**
# - **applicable source/family**: output-specific
# - **implementation module**: `future urp4.ensemble`
# - **validation evidence**: TRAIN-4TH/5TH source audit only
# - **blocking condition**: validated method comparison and holdout absent
# - **related work ID**: `HQ-ENSEMBLE-001`
# - **next evidence required**: Per-output selection and ensemble validation
from urp4.hq.v0_2 import stage_view
STAGE_17 = stage_view(REGISTRY, 'HQ-CELL-17')
STAGE_17


In [ ]:
# # HQ-CELL-18 — Forward pipeline
# - **semantic cell ID**: `HQ-CELL-18`
# - **stage name**: Forward pipeline
# - **purpose**: Expose θ→geometry→x→y while reporting the actual state of each link.
# - **input**: Generation θ, qualified geometry/descriptor/model contracts
# - **output**: Forward status/result when all links are approved
# - **current status**: **EXPERIMENTAL**
# - **applicable source/family**: family/source/output-specific
# - **implementation module**: `generators + descriptor service + future model service`
# - **validation evidence**: θ→geometry and geometry→minimal x technical evidence; y link locked
# - **blocking condition**: official y/model and broad descriptor validation absent
# - **related work ID**: `HQ-FWD-001`
# - **next evidence required**: Complete validated x→y link
from urp4.hq.v0_2 import stage_view
STAGE_18 = stage_view(REGISTRY, 'HQ-CELL-18')
STAGE_18


In [ ]:
# # HQ-CELL-19 — Inverse design
# - **semantic cell ID**: `HQ-CELL-19`
# - **stage name**: Inverse design
# - **purpose**: Represent target y*→x*→θ*→geometry without claiming success.
# - **input**: Approved forward models, constraints and uncertainty
# - **output**: Candidate design package
# - **current status**: **LOCKED**
# - **applicable source/family**: future validated domains
# - **implementation module**: `future urp4.inverse_design`
# - **validation evidence**: Concept/roadmap only
# - **blocking condition**: forward model and design-space validation absent
# - **related work ID**: `HQ-INV-001`
# - **next evidence required**: Forward evidence, optimization contract and physical validation
from urp4.hq.v0_2 import stage_view
STAGE_19 = stage_view(REGISTRY, 'HQ-CELL-19')
STAGE_19


In [ ]:
# # HQ-CELL-20 — Export, manifest and report
# - **semantic cell ID**: `HQ-CELL-20`
# - **stage name**: Export, manifest and report
# - **purpose**: Export only artifacts that actually exist with hashes and QA.
# - **input**: Completed stage outputs
# - **output**: Versioned artifacts/config/manifests/report
# - **current status**: **EXPERIMENTAL**
# - **applicable source/family**: all executed stages
# - **implementation module**: `HQ v0.1 exporter + future composite exporter`
# - **validation evidence**: HQ v0.1 manifests and clean package
# - **blocking condition**: later X/y/model/design artifacts unavailable
# - **related work ID**: `HQ-EXPORT-001`
# - **next evidence required**: Extend exporter only as stages are unlocked
from urp4.hq.v0_2 import stage_view
STAGE_20 = stage_view(REGISTRY, 'HQ-CELL-20')
STAGE_20


In [ ]:
# # HQ-CELL-21 — Final gate and handoff
# - **semantic cell ID**: `HQ-CELL-21`
# - **stage name**: Final gate and handoff
# - **purpose**: Summarize passed/failed/quarantined/incomplete stages and next work ID.
# - **input**: Stage registry + run artifacts + QA
# - **output**: Handoff/merge recommendation
# - **current status**: **READY**
# - **applicable source/family**: all
# - **implementation module**: `blueprint stage registry + report`
# - **validation evidence**: HQ-BLUEPRINT-001 verifier
# - **blocking condition**: scientific stages stay individually gated
# - **related work ID**: `HQ-BLUEPRINT-001`
# - **next evidence required**: IMSTL-008 remains next scientific run
from urp4.hq.v0_2 import stage_view
STAGE_21 = stage_view(REGISTRY, 'HQ-CELL-21')
STAGE_21
